# Ollama Prompt Playground

A minimal notebook that:

1. Reads models from **`models.toml`**
2. Reads versioned prompts from **`prompts/`** (files named `<name>_v<version>.md`, e.g. `greet_v1.md`, `greet_v2.md`)
3. Serves a Gradio UI where you pick a model, a prompt and a version, tweak the prompt text, and stream the reply from your local Ollama server.
4. Runs **`evaluation()`** from **`evaluation.py`** (a file outside the notebook): paste into the left box, click Evaluate, the result appears in the right box. Edit `evaluation.py` and hit *Reload evaluation.py* to apply changes without restarting.

**Setup**

```bash
pip install gradio requests
ollama serve          # if Ollama isn't already running
ollama pull llama3.1  # or any model you list in models.toml
```

- Run the cells from the folder that should contain `models.toml` and `prompts/`.
- **First run**: The notebook automatically calls `setup_prompt_eval()` to create default `models.toml`, `prompts/`, and `evaluation.py` files.
- Or run standalone: `python setup.py`
- Edit `models.toml` / add `_vN.md` files, then restart the kernel to reload them.


In [1]:
import importlib
import json
import re
from pathlib import Path

import tomllib  # Python 3.11+. On older Pythons: pip install tomli  then  import tomli as tomllib

import gradio as gr
import requests

BASE_DIR = Path.cwd()
MODELS_TOML = BASE_DIR / "models.toml"
PROMPTS_DIR = BASE_DIR / "prompts"


In [2]:
# --- Setup: initialize files if they don't exist ---
from setup import setup_prompt_eval
setup_prompt_eval()

# Import evaluation module now that files are initialized
import evaluation as evaluation_module


Created models.toml
Created prompts/greet_v1.md
Created prompts/greet_v2.md
Created prompts/summarise_v1.md


In [3]:
# --- Load model config from models.toml ---

def load_config(path=MODELS_TOML):
    with open(path, "rb") as f:
        return tomllib.load(f)

CONFIG = load_config()
OLLAMA_URL = CONFIG.get("ollama", {}).get("url", "http://localhost:11434")
MODELS = CONFIG.get("models", {})

print(f"Ollama server: {OLLAMA_URL}")
for name, cfg in MODELS.items():
    print(f"  {name} -> {cfg.get('model_id')} (temp={cfg.get('temperature', 0.7)})")


Ollama server: http://localhost:11434
  mistral:latest -> mistral:latest (temp=0.5)
  gemma4:12b -> gemma4:12b (temp=0.6)
  gpt-oss-20b-32k -> gpt-oss-20b-32k:latest (temp=0.6)
  qwen2.5-coder:7b -> qwen2.5-coder:7b (temp=0.2)
  qwen3.8:27b -> qwen3.8:27b (temp=0.5)
  glm-4.7-flash:latest -> glm-4.7-flash:latest (temp=0.7)
  deepseek-v2:16b -> deepseek-v2:16b (temp=0.5)


In [4]:
# --- Load versioned prompts from prompts/ ---
# Files must be named <name>_v<version>.md, e.g. greet_v1.md, greet_v2.md.

PROMPT_FILE = re.compile(r"^(?P<name>.+)_v(?P<version>\d+)$")

def load_prompts(directory=PROMPTS_DIR):
    prompts = {}
    for file in sorted(directory.glob("*.md")):
        m = PROMPT_FILE.match(file.stem)
        if not m:
            print(f"Skipping (unexpected name): {file.name}")
            continue
        prompts.setdefault(m["name"], {})[int(m["version"])] = (
            file.read_text(encoding="utf-8").strip()
        )
    return prompts

PROMPTS = load_prompts()
for name, versions in PROMPTS.items():
    print(name, "->", ", ".join(f"v{v}" for v in sorted(versions)))


greet -> v1, v2
summarise -> v1


In [5]:
# --- Ollama call + UI event handlers ---

def call_ollama(model_name, prompt_text, user_message, max_tokens):
    """Stream a chat completion; yields the growing text so Gradio updates live."""
    cfg = MODELS.get(model_name, {})
    payload = {
        "model": cfg.get("model_id", model_name),
        "messages": [
            {"role": "system", "content": prompt_text},
            {"role": "user", "content": user_message},
        ],
        "stream": True,
        "options": {
            "temperature": cfg.get("temperature", 0.7),
            "num_ctx": cfg.get("num_ctx", 4096),
            "num_predict": int(max_tokens),
        },
    }
    header = "**User:**\n> " + user_message + "\n\n**Assistant:**\n\n"
    text = ""
    with requests.post(f"{OLLAMA_URL}/api/chat", json=payload, stream=True, timeout=300) as resp:
        resp.raise_for_status()
        for line in resp.iter_lines():
            if not line:
                continue
            data = json.loads(line)
            if "error" in data:
                raise RuntimeError(data["error"])
            text += data.get("message", {}).get("content", "")
            yield header + text


def on_prompt_change(prompt_name):
    """When the prompt dropdown changes, refresh versions and show the latest text."""
    versions = sorted(PROMPTS.get(prompt_name, {}), reverse=True)
    if not versions:
        return gr.update(choices=[], value=None), ""
    return (
        gr.update(choices=[str(v) for v in versions], value=str(versions[0])),
        PROMPTS[prompt_name][versions[0]],
    )


def on_version_change(prompt_name, version):
    """When the version dropdown changes, show that version's text."""
    if prompt_name and version:
        return PROMPTS.get(prompt_name, {}).get(int(version), "")
    return ""


def run_evaluation(pasted_text):
    """Left box content -> external evaluation() from evaluation.py."""
    return evaluation_module.evaluation(pasted_text)


def reload_evaluation():
    """Re-import evaluation.py so edits take effect without a kernel restart."""
    importlib.reload(evaluation_module)
    return "*evaluation.py reloaded*"


In [6]:
# --- Gradio UI ---

with gr.Blocks(title="Ollama Prompt Playground") as demo:
    gr.Markdown(
        "### Ollama Prompt Playground\n"
        "Model from `models.toml`, prompt from `prompts/`. "
        "Reload cells above to pick up new files."
    )
    with gr.Row():
        with gr.Column():
            with gr.Row():
                model_dd = gr.Dropdown(choices=list(MODELS), value=next(iter(MODELS), None), label="Model")
                prompt_dd = gr.Dropdown(choices=list(PROMPTS), value=next(iter(PROMPTS), None), label="Prompt")
                version_dd = gr.Dropdown(choices=[], label="Version")
            prompt_tb = gr.Textbox(label="Prompt (editable)", lines=6)
            user_tb = gr.Textbox(label="Your message", lines=3)
            with gr.Row():
                max_tokens = gr.Slider(128, 4096, value=1024, step=128, label="Max tokens")
                run_btn = gr.Button("Run", variant="primary")
        with gr.Column():
            output_md = gr.Markdown(label="Chat", value="**User:** (your message will appear here)\n\n**Assistant:** ...")

    prompt_dd.change(on_prompt_change, inputs=prompt_dd, outputs=[version_dd, prompt_tb])
    version_dd.change(on_version_change, inputs=[prompt_dd, version_dd], outputs=prompt_tb)
    run_btn.click(call_ollama, inputs=[model_dd, prompt_tb, user_tb, max_tokens], outputs=output_md)

    # Initialise the version dropdown + prompt text on page load
    demo.load(on_prompt_change, inputs=prompt_dd, outputs=[version_dd, prompt_tb])

    # --- Evaluation: paste on the left, evaluation() result on the right ---
    gr.Markdown("---\n### Evaluation")
    eval_status = gr.Markdown()
    with gr.Row():
        with gr.Column():
            eval_input = gr.Textbox(label="Paste here", lines=10,
                                    placeholder="Paste anything you want evaluated...")
            with gr.Row():
                eval_btn = gr.Button("Evaluate", variant="primary")
                reload_btn = gr.Button("Reload evaluation.py", size="sm")
        with gr.Column():
            eval_output = gr.Textbox(label="Result", lines=10, interactive=False)

    eval_btn.click(run_evaluation, inputs=eval_input, outputs=eval_output)
    reload_btn.click(reload_evaluation, outputs=eval_status)

demo.launch()  # add share=True for a temporary public link


* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.
